# Module 2 · GET & POST Requests, and Serving an ML Model
### FastAPI for ML Engineers — Bootcamp Session (Part 2 of 4)

**Time budget:** ~60 minutes
**Prereq:** Module 1 (API fundamentals & auth)

## Agenda
| Time | Topic |
|---|---|
| 0:00–0:10 | GET vs POST — semantics and why it matters for ML APIs |
| 0:10–0:20 | Request/response schemas with Pydantic |
| 0:20–0:45 | Building a real `/predict` endpoint around a trained model |
| 0:45–0:55 | Batch prediction, async, error handling |
| 0:55–1:00 | Exercise + wrap-up |

## Learning objectives
- Know when to use `GET` vs `POST` (and why `GET` is *wrong* for most model inference)
- Define strict input/output contracts with Pydantic `BaseModel`
- Wrap a trained scikit-learn model behind a FastAPI service
- Handle validation errors and model errors gracefully with correct status codes
- Support batch requests and understand when `async def` actually helps

In [ ]:
# If running this notebook fresh (new session), make sure deps are installed:
!pip install -q --break-system-packages fastapi "uvicorn[standard]" httpx requests scikit-learn joblib numpy pandas
print("done")

## 2.1 GET vs POST — and why it matters for ML

| | `GET` | `POST` |
|---|---|---|
| Purpose | Retrieve a resource | Create / submit / process something |
| Body | No body (by convention/spec — many servers ignore or reject one) | Has a body |
| Idempotent? | Yes — calling it twice has no extra effect | Not necessarily |
| Cacheable? | Yes, by default | No, by default |
| Parameters | Query string / path only | JSON body (usually) |
| Visible in URL / logs / browser history | Yes | No (body isn't logged the same way) |

### Why `POST` is (almost always) the right choice for `/predict`
Model inputs are frequently:
- **Too large or complex** for a query string (feature vectors, arrays, images, text blobs)
- **Sensitive** (PII, account data) — you don't want that sitting in a URL, server access logs, or browser history
- **Not idempotent in effect** in the sense that you might log/charge/rate-limit per call, even though the *computation* is deterministic

So the convention across virtually every ML serving framework (Vertex AI, SageMaker, TorchServe, Seldon, KServe) is:
- `GET /health`, `GET /` → liveness/readiness checks, metadata
- `POST /predict` (or `/v1/models/{model}:predict` in Vertex's case) → actual inference

We'll build exactly that shape below.

## 2.2 Pydantic schemas — your input/output contract

FastAPI uses **Pydantic** models to validate incoming JSON and to document your API automatically. Instead of manually checking `if "feature_1" not in body: raise ...`, you declare the *shape* once and FastAPI enforces it — wrong types, missing fields, or extra junk fields get rejected with a `422` before your model code ever runs.

This matters a lot for ML: garbage input silently reaching `model.predict()` is how you get either a crash or, worse, a *garbage prediction that looks valid*.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List

class PredictionRequest(BaseModel):
    feature_1: float = Field(..., description="Account age in years")
    feature_2: float = Field(..., description="Number of transactions in last 24h")
    feature_3: float = Field(..., description="Average transaction amount")

    @field_validator("feature_2")
    @classmethod
    def non_negative_txn_count(cls, v):
        if v < 0:
            raise ValueError("feature_2 (transaction count) cannot be negative")
        return v

class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    model_version: str

# quick sanity check outside of FastAPI entirely — Pydantic validates on construction
try:
    bad = PredictionRequest(feature_1=2.0, feature_2=-5, feature_3=10.0)
except Exception as e:
    print("Validation caught it:", e)

good = PredictionRequest(feature_1=2.0, feature_2=5, feature_3=10.0)
print(good)

## 2.3 Train a tiny model to serve

We're not here to build a great model — a small, fast one lets us focus entirely on the serving layer. We'll train a logistic regression "fraud-ish" classifier on synthetic data and pickle it with `joblib`, exactly like you'd do before containerizing it in Module 3.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import joblib, os

X, y = make_classification(n_samples=2000, n_features=3, n_informative=3,
                            n_redundant=0, random_state=42)

clf = LogisticRegression()
clf.fit(X, y)

os.makedirs("/home/claude/model_artifacts", exist_ok=True)
MODEL_PATH = "/home/claude/model_artifacts/model.joblib"
joblib.dump(clf, MODEL_PATH)

print("train accuracy:", clf.score(X, y))
print("saved to:", MODEL_PATH)

## 2.4 Wrapping the model in FastAPI

Key design choices worth calling out (these are the things interviewers and code reviewers actually check):

1. **Load the model once at startup**, not on every request — use FastAPI's `lifespan` (or the older `@app.on_event("startup")`) so cold-start cost is paid once.
2. **Validate with Pydantic**, catch bad input before it reaches the model.
3. **Wrap model inference in try/except** and return `500` (or a domain-specific error) instead of letting the process crash or leak a stack trace.
4. **Return structured output**, not a bare number — include a model version so consumers can track what produced a prediction.

In [ ]:
from fastapi import FastAPI, HTTPException
from contextlib import asynccontextmanager
import joblib
import numpy as np

ML_MODELS = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # startup: load once
    ML_MODELS["fraud_model"] = joblib.load(MODEL_PATH)
    print("model loaded")
    yield
    # shutdown: clean up if needed
    ML_MODELS.clear()

ml_app = FastAPI(title="Fraud Model Serving API", version="1.0.0", lifespan=lifespan)

MODEL_VERSION = "fraud_v1_logreg"

@ml_app.get("/health")
def health():
    return {"status": "ok", "model_loaded": "fraud_model" in ML_MODELS}

@ml_app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    model = ML_MODELS.get("fraud_model")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")

    features = np.array([[request.feature_1, request.feature_2, request.feature_3]])

    try:
        pred = int(model.predict(features)[0])
        proba = float(model.predict_proba(features)[0][pred])
    except Exception as e:
        # never let a raw stack trace leak to the client
        raise HTTPException(status_code=500, detail=f"Inference failed: {type(e).__name__}")

    return PredictionResponse(prediction=pred, probability=proba, model_version=MODEL_VERSION)

In [ ]:
from fastapi.testclient import TestClient

# TestClient triggers the lifespan startup/shutdown for us when used as a context manager
with TestClient(ml_app) as client:
    print(client.get("/health").json())

    good_req = {"feature_1": 2.0, "feature_2": 5, "feature_3": 10.0}
    resp = client.post("/predict", json=good_req)
    print(resp.status_code, resp.json())

    # Missing field -> FastAPI/Pydantic auto-generates a 422
    resp = client.post("/predict", json={"feature_1": 2.0})
    print("missing field ->", resp.status_code)
    print(resp.json())

    # Our custom validator -> also a 422, with our message inside
    resp = client.post("/predict", json={"feature_1": 2.0, "feature_2": -1, "feature_3": 3.0})
    print("negative txn count ->", resp.status_code)
    print(resp.json())

Notice: we never had to write `if "feature_1" not in body`. FastAPI + Pydantic handled missing fields, wrong types, *and* our custom business rule (non-negative transaction count) — all producing a clean `422` with a machine-readable error body, not a crash.

## 2.5 Batch prediction

Real traffic is rarely one row at a time. A batch endpoint amortizes overhead (network round-trips, occasionally model call overhead) and is what most ML platforms expect for anything beyond trivial QPS.

In [ ]:
class BatchPredictionRequest(BaseModel):
    instances: List[PredictionRequest]

class BatchPredictionResponse(BaseModel):
    predictions: List[PredictionResponse]

@ml_app.post("/predict/batch", response_model=BatchPredictionResponse)
def predict_batch(request: BatchPredictionRequest):
    model = ML_MODELS.get("fraud_model")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")

    if len(request.instances) == 0:
        raise HTTPException(status_code=400, detail="instances must not be empty")

    features = np.array([[r.feature_1, r.feature_2, r.feature_3] for r in request.instances])

    preds = model.predict(features)
    probas = model.predict_proba(features)

    results = [
        PredictionResponse(
            prediction=int(p),
            probability=float(probas[i][p]),
            model_version=MODEL_VERSION,
        )
        for i, p in enumerate(preds)
    ]
    return BatchPredictionResponse(predictions=results)

In [ ]:
with TestClient(ml_app) as client:
    batch_req = {
        "instances": [
            {"feature_1": 2.0, "feature_2": 5, "feature_3": 10.0},
            {"feature_1": -1.0, "feature_2": 40, "feature_3": 2.0},
            {"feature_1": 0.5, "feature_2": 1, "feature_3": 50.0},
        ]
    }
    resp = client.post("/predict/batch", json=batch_req)
    print(resp.status_code)
    for r in resp.json()["predictions"]:
        print(r)

## 2.6 `async def` vs `def` — when does it actually matter?

FastAPI lets you declare routes as either `def` or `async def`. A common misconception is "async is always faster." The reality:

- If your handler is **CPU-bound** (most `model.predict()` calls for classical ML — sklearn, XGBoost, LightGBM), a plain `def` is usually *fine or better*: FastAPI runs sync `def` endpoints in a thread pool, so one slow request doesn't block the whole event loop.
- If your handler does **I/O-bound work** — calling another API, hitting a feature store over the network, reading from a database — use `async def` and `await` those calls. This is where async actually buys you concurrency.
- For a **GPU-bound deep learning model** behind a queue/batching layer (e.g., Triton, TorchServe), you'd typically make the FastAPI route `async def` and `await` a call into that inference server, rather than doing the heavy compute inline.

Rule of thumb: **don't mark a route `async def` and then call blocking code inside it** (e.g., `time.sleep`, a blocking DB driver, `requests.get`) — that *does* block the event loop and can be worse than plain `def`. If you need blocking calls inside an async route, run them in a thread pool via `run_in_threadpool` / `asyncio.to_thread`.

In [ ]:
import asyncio
import httpx

@ml_app.get("/downstream-example")
async def call_downstream():
    # Example: async route doing real I/O — this is where async helps.
    # (This specific call will fail in a sandboxed notebook with no network access —
    #  that's expected; the point is the pattern, not this exact URL succeeding.)
    async with httpx.AsyncClient(timeout=2.0) as http_client:
        try:
            r = await http_client.get("https://example.com")
            return {"downstream_status": r.status_code}
        except Exception as e:
            return {"downstream_status": None, "error": type(e).__name__}

In [ ]:
with TestClient(ml_app) as client:
    resp = client.get("/downstream-example")
    print(resp.status_code, resp.json())

## 2.7 Calling your API like a client would (`requests`)

Everything above used `TestClient`, which is perfect for automated tests but only works in-process. Once you actually run the service with `uvicorn app:app --host 0.0.0.0 --port 8000`, any external client — including your own teammates, a frontend, or another microservice — talks to it exactly like this:

```python
import requests

resp = requests.post(
    "http://localhost:8000/predict",
    json={"feature_1": 2.0, "feature_2": 5, "feature_3": 10.0},
    headers={"X-API-Key": "supersecret-demo-key"},  # if you added auth from Module 1
)
print(resp.status_code, resp.json())
```

This is exactly the shape your Docker container (Module 3) and your Vertex AI endpoint (Module 4) will be called with.

## Exercise 2 (10 min)

Extend `ml_app` so that:
1. `/predict` accepts an optional `explain: bool = False` query parameter. When `True`, also return each feature's contribution (for logistic regression, `feature * coefficient` is good enough — no need for full SHAP).
2. Add input bounds checking: reject (`400`) any request where `feature_3` (transaction amount) is negative or above `100000`.
3. Bonus: add a `request_id` (use `uuid.uuid4()`) to every response so you could correlate it with logs later.

Scaffold below.

In [ ]:
import uuid

@ml_app.post("/predict/v2")
def predict_v2(request: PredictionRequest, explain: bool = False):
    model = ML_MODELS.get("fraud_model")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")

    # TODO: bounds check feature_3 (0 <= feature_3 <= 100000), else raise HTTPException(400, ...)

    features = np.array([[request.feature_1, request.feature_2, request.feature_3]])
    pred = int(model.predict(features)[0])
    proba = float(model.predict_proba(features)[0][pred])

    result = {
        "prediction": pred,
        "probability": proba,
        "model_version": MODEL_VERSION,
        "request_id": str(uuid.uuid4()),
    }

    # TODO: if explain=True, add result["contributions"] = {feature_name: feature_value * coef}
    #       (model.coef_[0] gives you the 3 coefficients in feature order)

    return result

# Try it once implemented:
# with TestClient(ml_app) as client:
#     print(client.post("/predict/v2?explain=true", json=good_req).json())

## Recap

- `GET` for retrieval/health checks, `POST` for inference — this is the near-universal convention across serving frameworks.
- Pydantic models are your validation layer *and* your documentation — define them tightly.
- Load models once at startup (`lifespan`), never per-request.
- Wrap inference in try/except and return meaningful status codes; never let a raw exception reach the client.
- Batch endpoints matter for throughput; `async def` only helps for I/O-bound work, not CPU-bound model inference.

**Next up — Module 3:** packaging this exact service into a Docker image so it can run anywhere — including Vertex AI in Module 4.